# HackTheFork — benchmark des modèles

Comparer les modèles sur le **prix minimum permettant à la marge lentille d’égaler ou dépasser celle du blé dans 80 % des scénarios**.

Sélectionner un **GPU T4**, puis **Exécution → Tout exécuter**. Autoriser Drive et saisir la clé Prior Labs dans le champ masqué (ou créer le secret Colab `TABPFN_TOKEN`).

Deux barres suivent les modèles de rendement puis les modèles directs du prix, avec le modèle et l’année en cours. Chaque étape terminée est sauvegardée dans **Mon Drive / HackTheFork / benchmarks**. Après une coupure, réexécuter les cellules : seules les étapes inachevées sont recalculées. Une étape correspond à un modèle et une année, calibration comprise ; sa durée peut varier.

Avant le premier lancement, accepter la licence **TabPFN 3.5** dans [Prior Labs → Licenses](https://ux.priorlabs.ai/account/licenses). Le script vérifie la clé et cette licence avant les entraînements.


In [ ]:
from google.colab import drive, userdata
from getpass import getpass
import os

REMPLACER_CLE = False  # True pour saisir une nouvelle clé après un refus d'accès.
drive.mount('/content/drive')
if REMPLACER_CLE:
    os.environ['TABPFN_TOKEN'] = getpass('Nouvelle clé API Prior Labs : ').strip()
else:
    try:
        # Relire le secret pour ne pas conserver une ancienne clé en mémoire.
        os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN').strip()
    except Exception:
        if not os.environ.get('TABPFN_TOKEN', '').strip():
            os.environ['TABPFN_TOKEN'] = getpass('Clé API Prior Labs : ').strip()
if not os.environ.get('TABPFN_TOKEN', '').strip():
    raise ValueError('La clé Prior Labs est nécessaire pour TabPFN.')
os.environ['TABPFN_NO_BROWSER'] = '1'
print('Drive connecté et clé chargée. La validité et la licence seront vérifiées avant le benchmark.')


## Préparer ou retrouver l’expérience

Les sources sont conservées dans Drive pour reprendre exactement la même expérience, même si GitHub évolue. Mettre `NOUVELLE_VERSION = True` pour récupérer une nouvelle version ; les résultats précédents restent disponibles.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
import requests

NOUVELLE_VERSION = False
REPO = 'nassim-arifette/hackthefork-hackathon'
REF = 'research/probabilistic-benchmark'
folder = Path('/content/drive/MyDrive/HackTheFork')
folder.mkdir(parents=True, exist_ok=True)
source_archive = folder / 'benchmark-source.zip'
if NOUVELLE_VERSION or not source_archive.exists():
    response = requests.get(f'https://api.github.com/repos/{REPO}/commits/{REF}', timeout=60)
    response.raise_for_status()
    commit = response.json()['sha']
    response = requests.get(f'https://codeload.github.com/{REPO}/zip/{commit}', timeout=180)
    response.raise_for_status()
    temporary = source_archive.with_suffix('.zip.tmp')
    temporary.write_bytes(response.content)
    with zipfile.ZipFile(temporary) as z:
        if z.testzip() is not None:
            raise ValueError('Archive incomplète : relancer la cellule.')
    temporary.replace(source_archive)

source_root = Path('/content/hackthefork-source')
source_root.mkdir(exist_ok=True)
with zipfile.ZipFile(source_archive) as z:
    for item in z.infolist():
        if not (source_root / item.filename).resolve().is_relative_to(source_root.resolve()):
            raise ValueError('Chemin archive invalide')
    z.extractall(source_root)
    project = source_root / z.namelist()[0].split('/')[0]
subprocess.run([sys.executable, str(project / 'scripts/package_benchmark.py')], check=True)
shutil.copy2(project / 'benchmark-input.zip', '/content/benchmark-input.zip')
print('Sources prêtes :', project.name)


In [ ]:
import runpy
_ = runpy.run_path(str(project / 'scripts/colab_benchmark.py'), run_name='__main__')


Le ZIP final est enregistré automatiquement dans le dossier de l’expérience sur Drive. Les fichiers `predictions.csv` donnent les résultats par département et année ; `results.json` contient les scores et les paramètres. Les points de reprise conservent les résultats et les erreurs de calibration, sans enregistrer la clé API. Ils ne sauvegardent pas un entraînement au milieu d’une étape.

**Si une ancienne exécution s’arrête à TabPFN :** accepter la licence 3.5 sur le compte de la clé, remplacer la variable `TABPFN_TOKEN` en mémoire si nécessaire, puis relancer sa dernière cellule. Conserver `NOUVELLE_VERSION = False` et l’archive existante pour reprendre les mêmes checkpoints. Ne pas changer de version uniquement pour renouveler une clé.
